## Topic: Hyperparameter Tuning with Pipeline (GridSearchCV)
### Goal of Today

By the end of this session, we will:

- Tune models automatically
- Combine Pipeline + GridSearch
- Find best model settings
- Build optimized ML systems

### 1. What is the Problem?

👉 Until now:
```
Model parameters = manually chosen
```
Example:
```
LogisticRegression(C=1)
```
👉 But is C=1 best?

👉 We don’t know.

### 2. Solution: GridSearchCV

👉 Automatically tries multiple combinations

Concept:
```
Try → Evaluate → Select Best
```
### Tools You Will Use
- Scikit-learn
- Pipeline
- GridSearchCV


Now we combine Pipeline + Cross-Validation + GridSearchCV into one full workflow and explain every line with comments so the concept becomes crystal clear.

We’ll reuse a fake dataset so you can run everything.

### Step 0 — Import Libraries

In [4]:
# Create dataset
from sklearn.datasets import make_classification

# Split data
from sklearn.model_selection import train_test_split

# Pipeline tools
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

### Step 1 — Create Dataset

In [5]:
# Generate fake business dataset
X, y = make_classification(
    n_samples=1000,   # rows
    n_features=5,     # columns
    random_state=42
)

# Split into train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

We now have unseen test data for final evaluation.

### Step 2 — Create Pipeline

In [6]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),        # Step 1: scale features
    ('model', LogisticRegression())      # Step 2: model
])

Meaning

Pipeline = automatic workflow:
```
Data → Scaling → Model → Prediction
```
No manual preprocessing needed.

### Step 3 — Define Parameter Grid

In [7]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['liblinear', 'lbfgs']
}

Important Concept

Inside pipeline we access parameters using:
```
stepName__parameter
```
So:
| Key           | Meaning                     |
| ------------- | --------------------------- |
| model__C      | Change LogisticRegression C |
| model__solver | Try different solvers       |

### Step 4 — Apply GridSearchCV

In [8]:
grid = GridSearchCV(
    pipeline,      # pipeline to tune
    param_grid,    # combinations to try
    cv=5           # 5-fold cross validation
)

What GridSearch will do

It will try ALL combinations:
| C    | Solver    |
| ---- | --------- |
| 0.01 | liblinear |
| 0.01 | lbfgs     |
| 0.1  | liblinear |
| 0.1  | lbfgs     |
| 1    | liblinear |
| 1    | lbfgs     |
| 10   | liblinear |
| 10   | lbfgs     |

👉 8 models × 5 folds = 40 trainings automatically

### Step 5 — Train Grid Search

In [9]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', LogisticRegression())]),
             param_grid={'model__C': [0.01, 0.1, 1, 10],
                         'model__solver': ['liblinear', 'lbfgs']})

Behind the scenes:

For each combination:
```
Scale data → Train → Cross-validate → Score
```
Everything automated

### Step 6 — See Best Parameters

Example output:
```
Best Parameters: {'model__C': 0.1, 'model__solver': 'liblinear'}
Best CV Score: 0.86
```

In [10]:
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'model__C': 0.01, 'model__solver': 'liblinear'}
Best CV Score: 0.8574999999999999


Meaning:

👉 Out of 40 trainings, this combo performed best.

### Step 7 — Use Best Model
```
best_model = grid.best_estimator_
```

This gives us a fully trained pipeline with best settings.

Predict on test data:

In [12]:
best_model = grid.best_estimator_

In [13]:
predictions = best_model.predict(X_test)

print("Test Accuracy:", best_model.score(X_test, y_test))

Test Accuracy: 0.865


What Just Happened (Big Picture)

System automatically:

1. Tried many models
2. Used cross-validation
3. Found best settings
4. Returned optimized pipeline

This is how real ML systems are built.

### Step 8 — Tune Decision Tree (Second Example)

Now we change model easily.

In [14]:
pipeline_tree = Pipeline([
    ('model', DecisionTreeClassifier())
])

Notice:

No scaler needed for trees.

Define Tree Parameters

In [15]:
param_grid_tree = {
    'model__max_depth': [2, 3, 5, 10],
    'model__min_samples_split': [2, 5, 10]
}

Grid will try:

4 depths × 3 splits = 12 combinations

Run Grid Search

In [16]:
grid_tree = GridSearchCV(
    pipeline_tree,
    param_grid_tree,
    cv=5
)

grid_tree.fit(X_train, y_train)

print("Best Tree Params:", grid_tree.best_params_)
print("Best Tree Score:", grid_tree.best_score_)

Best Tree Params: {'model__max_depth': 5, 'model__min_samples_split': 5}
Best Tree Score: 0.9012499999999999


### Why Pipeline + GridSearch Together?
#### Without Pipeline

Scaling happens before split → data leakage

Model cheats by seeing test data stats.

#### With Pipeline

Scaling happens inside CV folds → safe.

This is industry best practice.

### MASTER CONCEPT

Real ML workflow:
```
Raw Data
   ↓
Pipeline
   ↓
Cross-Validation
   ↓
GridSearchCV
   ↓
Best Model
   ↓
Deployment
```
You now know the professional ML tuning workflow